<a href="https://colab.research.google.com/github/lindseyramsey1/dsa405-project/blob/main/DSA405_002_FA26_P2_laramse3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P2: Data Audit, Cleaning Log & Provenance Brief

### DSA 405 · Fall 2026 · Project Milestone 2

#### Lindsey Ramsey



In [2]:
import pandas as pd
import numpy as np
import requests

## 1. Data Audit

For Part 2, I audited the Bureau of Labor Statistics Current Employment
Statistics wage dataset used in my project. Before making any changes, I
examined factors like structure of the raw data, data types, missing values, distinct
values, numeric ranges, categorical values, duplicate records, monthly
coverage, possible sentinel values, identifier formatting, and text encoding.

In [3]:
series_ids = {"Construction": "CES2000000003", "Manufacturing": "CES3000000003", "Trade, Transportation, and Utilities": "CES4000000003", "Information": "CES5000000003", "Financial Activities": "CES5500000003", "Professional and Business Services": "CES6000000003", "Private Education and Health Services": "CES6500000003", "Leisure and Hospitality": "CES7000000003"}

series_ids

{'Construction': 'CES2000000003',
 'Manufacturing': 'CES3000000003',
 'Trade, Transportation, and Utilities': 'CES4000000003',
 'Information': 'CES5000000003',
 'Financial Activities': 'CES5500000003',
 'Professional and Business Services': 'CES6000000003',
 'Private Education and Health Services': 'CES6500000003',
 'Leisure and Hospitality': 'CES7000000003'}

In [35]:
url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
payload_1 = {"seriesid": list(series_ids.values()), "startyear": "2015", "endyear": "2024"}
payload_2 = {"seriesid": list(series_ids.values()), "startyear": "2025", "endyear": "2025"}
response_1 = requests.post(url, json=payload_1)
response_2 = requests.post(url, json=payload_2)
bls_raw_1 = response_1.json()
bls_raw_2 = response_2.json()

print("Request 1:", bls_raw_1["status"])
print("Request 2:", bls_raw_2["status"])

Request 1: REQUEST_SUCCEEDED
Request 2: REQUEST_SUCCEEDED


In [36]:
id_to_industry = {v: k for k, v in series_ids.items()}
rows = []
for bls_raw in [bls_raw_1, bls_raw_2]:
    for series in bls_raw["Results"]["series"]:
        series_id = series["seriesID"]
        industry = id_to_industry[series_id]
        for obs in series["data"]:
            rows.append({"series_id": series_id, "industry": industry, "year": obs["year"], "period": obs["period"], "period_name": obs["periodName"], "hourly_earnings": obs["value"]})
bls_df = pd.DataFrame(rows)
bls_df.head(10)

,series_id,industry,year,period,period_name,hourly_earnings
0,CES2000000003,Construction,2024,M12,December,38.94
1,CES2000000003,Construction,2024,M11,November,38.77
2,CES2000000003,Construction,2024,M10,October,38.69
3,CES2000000003,Construction,2024,M09,September,38.55
4,CES2000000003,Construction,2024,M08,August,38.34
5,CES2000000003,Construction,2024,M07,July,38.29
6,CES2000000003,Construction,2024,M06,June,38.10
7,CES2000000003,Construction,2024,M05,May,38.04
8,CES2000000003,Construction,2024,M04,April,37.99
9,CES2000000003,Construction,2024,M03,March,37.77


### Raw Data

I created a separate copy of the data as retrieved from BLS before performing
any cleaning or conversions to ensure I have the original to reference. The audit checks below are utilizing the original version.

In [6]:
raw_bls = bls_df.copy()
print("Raw dataset shape:", raw_bls.shape)
display(raw_bls.head())

Raw dataset shape: (1056, 6)


,series_id,industry,year,period,period_name,hourly_earnings
0,CES2000000003,Construction,2024,M12,December,38.94
1,CES2000000003,Construction,2024,M11,November,38.77
2,CES2000000003,Construction,2024,M10,October,38.69
3,CES2000000003,Construction,2024,M09,September,38.55
4,CES2000000003,Construction,2024,M08,August,38.34


### Structure, Data Types, Missing Values, and Distinct Values

I began by examining how each variable was identified and compared that type with the type I found most logical for the future analysis. This is shown in the "intended type" section. I also calculated the count and percentage
of missing values and the number of distinct values in each column.

In [39]:
audit = pd.DataFrame({"dtype_as_loaded": raw_bls.dtypes.astype(str), "missing_count": raw_bls.isna().sum(), "missing_rate_percent": (raw_bls.isna().mean() * 100).round(2), "distinct_values": raw_bls.nunique(dropna=False)})
intended_types = {"series_id": "string", "industry": "string/category", "year": "int", "period": "string/category", "period_name": "string/category", "hourly_earnings": "float"}

audit["intended_dtype"] = pd.Series(intended_types)
audit = audit[["dtype_as_loaded", "intended_dtype", "missing_count", "missing_rate_percent", "distinct_values"]]

audit

,dtype_as_loaded,intended_dtype,missing_count,missing_rate_percent,distinct_values
series_id,object,string,0,0.0,8
industry,object,string/category,0,0.0,8
year,object,int,0,0.0,11
period,object,string/category,0,0.0,12
period_name,object,string/category,0,0.0,12
hourly_earnings,object,float,0,0.0,902


### Numeric Range and Conversion Check

The year and hourly_earnings variables were perceived as text type variables. Before changing them, I tested whether every value could actually be converted to a numeric value and then found the minimum, maximum, and range of each variable.

In [41]:
print("Year values that could not be converted:", numeric_audit["year"].isna().sum())
print("Hourly earnings values that could not be converted:", numeric_audit["hourly_earnings"].isna().sum())

Year values that could not be converted: 0
Hourly earnings values that could not be converted: 0


In [40]:
numeric_audit = pd.DataFrame({"year": pd.to_numeric(raw_bls["year"],errors="coerce"), "hourly_earnings": pd.to_numeric(raw_bls["hourly_earnings"], errors="coerce")})
numeric_summary = pd.DataFrame({"Min": numeric_audit.min(), "Max": numeric_audit.max(), "Range": (numeric_audit.max() - numeric_audit.min())})
numeric_summary

,Min,Max,Range
year,2015.00,2025.0,10.00
hourly_earnings,14.15,53.9,39.75


### Categorical Values

For categorical variables with fewer than 30 levels, I analyzed the full
set of distinct values.

In [10]:
categorical_cols = ["series_id", "industry", "period", "period_name"]

for col in categorical_cols:
    print(f"\n{col}")
    print("Number of distinct values:", raw_bls[col].nunique(dropna=False))
    print(sorted(raw_bls[col].dropna().unique()))


series_id
Number of distinct values: 8
['CES2000000003', 'CES3000000003', 'CES4000000003', 'CES5000000003', 'CES5500000003', 'CES6000000003', 'CES6500000003', 'CES7000000003']

industry
Number of distinct values: 8
['Construction', 'Financial Activities', 'Information', 'Leisure and Hospitality', 'Manufacturing', 'Private Education and Health Services', 'Professional and Business Services', 'Trade, Transportation, and Utilities']

period
Number of distinct values: 12
['M01', 'M02', 'M03', 'M04', 'M05', 'M06', 'M07', 'M08', 'M09', 'M10', 'M11', 'M12']

period_name
Number of distinct values: 12
['April', 'August', 'December', 'February', 'January', 'July', 'June', 'March', 'May', 'November', 'October', 'September']


### Duplicate Checks

I checked for exact duplicate rows and duplicate observation keys. Each monthly observation should and needs to be uniquely identified by its BLS series ID, year,
and monthly period.

In [42]:
exact_duplicates = raw_bls.duplicated().sum()
print("Exact duplicate rows:", exact_duplicates)
duplicate_keys = raw_bls.duplicated(subset=["series_id", "year", "period"],keep=False)
print("Rows involved in duplicate " "series-year-period keys:", duplicate_keys.sum())
raw_bls.loc[duplicate_keys].sort_values(["series_id", "year", "period"])

Exact duplicate rows: 0
Rows involved in duplicate series-year-period keys: 0


,series_id,industry,year,period,period_name,hourly_earnings


### Monthly Period Validation

BLS monthly observations use period codes M01 through M12. I compared
the observed codes with the complete expected set to identify unexpected codes or missing monthly period categories.

In [43]:
expected_periods = {f"M{i:02d}" for i in range(1, 13)}
observed_periods = set(raw_bls["period"].dropna().unique())
unexpected_periods = (observed_periods - expected_periods)
missing_period_codes = (expected_periods - observed_periods)

print("Unexpected Period Codes:", unexpected_periods)
print("Expected Period Codes Not Found:", missing_period_codes)

Unexpected Period Codes: set()
Expected Period Codes Not Found: set()


### Monthly Coverage

I also checked whether every BLS series contains all 12 monthly observations
for each year from 2015 through 2025.

In [45]:
coverage = (raw_bls.groupby(["series_id", "year"])["period"].nunique().reset_index(name="months_included"))
coverage["complete_year"] = (coverage["months_included"] == 12)
coverage

,series_id,year,months_included,complete_year
0,CES2000000003,2015,12,True
1,CES2000000003,2016,12,True
2,CES2000000003,2017,12,True
3,CES2000000003,2018,12,True
4,CES2000000003,2019,12,True
...,...,...,...,...
83,CES7000000003,2021,12,True
84,CES7000000003,2022,12,True
85,CES7000000003,2023,12,True
86,CES7000000003,2024,12,True


In [48]:
print("Industry years with < 12 months:", (~coverage["complete_year"]).sum())

Industry years with < 12 months: 0


### Earnings Validity and Sentinel-Value Checks

I checked hourly_earnings for missing values, values that could not be interpreted as numbers, nonpositive earnings, and obvious sentinel values that could represent missing information stored as a numeric value.

In [51]:
earnings_numeric = pd.to_numeric(raw_bls["hourly_earnings"], errors="coerce")
print("Missing Earnings:", raw_bls["hourly_earnings"].isna().sum())
print("Non-Numeric Earnings:", earnings_numeric.isna().sum())
print("Earnings <= 0:", (earnings_numeric <= 0).sum())
print("Possible Sentinel Values" "(999, 9999, -999):", earnings_numeric.isin([999, 9999, -999]).sum())

Missing Earnings: 0
Non-Numeric Earnings: 0
Earnings <= 0: 0
Possible Sentinel Values(999, 9999, -999): 0


### Identifier Check

The BLS series_id is an identifier, not an actual measurement. I
checked that the identifiers remained stored as text, that none were missing,
and that the expected eight series remained distinct.

In [52]:
print("series_id dtype:", raw_bls["series_id"].dtype)
print("Missing Series IDs:", raw_bls["series_id"].isna().sum())
print("Unique Series IDs:", raw_bls["series_id"].nunique())
print("\nSeries IDs:")
print(sorted(raw_bls["series_id"].unique()))

series_id dtype: object
Missing Series IDs: 0
Unique Series IDs: 8

Series IDs:
['CES2000000003', 'CES3000000003', 'CES4000000003', 'CES5000000003', 'CES5500000003', 'CES6000000003', 'CES6500000003', 'CES7000000003']


### Text Encoding Check



In [53]:
text_cols = ["industry", "period_name"]

for col in text_cols:
    encoding_issues = (raw_bls[col].astype(str).str.contains("\ufffd", regex=False).sum())
    print(f"{col} possible encoding issues:", encoding_issues)

industry possible encoding issues: 0
period_name possible encoding issues: 0


### Total and Subtotal Row Check

I checked the industry labels for records identified as totals or subtotals to further identify the actual observations.

In [54]:
total_subtotal_rows = (raw_bls["industry"].astype(str).str.contains(r"\b(total|subtotal)\b", case=False, regex=True))
print("Possible Total or Subtotal rows:", total_subtotal_rows.sum())
raw_bls.loc[total_subtotal_rows, ["series_id", "industry"]].drop_duplicates()

Possible Total or Subtotal rows: 0


/tmp/ipykernel_1399/1108095220.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  total_subtotal_rows = (raw_bls["industry"].astype(str).str.contains(r"\b(total|subtotal)\b", case=False, regex=True))


,series_id,industry


## Audit Findings

The raw BLS dataset containts 1,056 rows and 6 columns, representing monthly average hourly earnings for 8 industries from 2015 to 2025. The audit did not detect any missing values in any of the six variables.

There were two data type issues within my dataset. The variable year was found as an object, however, it is supposed to represent a date value. The next variable, hourly_earnings was found as an object as well, however, contains numeric dollar values. With one of the checks, I found that we can convert these values, instead of including missing values.

There were not duplicate rows, including the within the series_id. All eight series_ids held observations for every month of the year.

The observed hourly earnings had a range from $14.15 to $53.90 per hour. Missing, non-numeric, nonpositive, or sentinel values were found within the audit.

The 8 series_id values were kept as identifiers, and none were missing from the dataset.

Overall, the BLS source is well structured and will need minimal cleaning done to the dataset. The next necessary changes include converting year and hourly_earnings into the correct data types for the next analysis.

## 2. Data Dictionary

The data dictionary below documents the variables that will be kept or created in the cleaned/new BLS dataset. It includes each variable's intended data type, units, factor levels when applicable, valid range, interpretation of missing values, and other information needed to interpret the data.


In [58]:
data_dictionary = pd.DataFrame([
    ["series_id", "string", "Identifier", "8 selected BLS CES series IDs", "One of the 8 selected BLS CES series IDs", "0 - missing means the BLS series is unidentified", "BLS identifier for each industry series"],
    ["industry", "string/category", "Industry Category/Categorical Variable", "8 selected industries", "One of the 8 selected industries", "0 - missing means the industry is unidentified", "Industry name is connected with series_id"],
    ["year", "integer", "year", "Calender Year", "2015-2025 for this project", "0 - missing means the observation year is unknown", "Presented as text and converted to integer"],
    ["period", "string/category", "Monthly Period Code", "M01-M12", "M01-M12", "0 - missing means the monthly period is unknown", "Original BLS month code - M01=January through M12=December"],
    ["period_name", "string/category", "Calender Month", "January-December", "January-December", "0 - missing means the month name is unknown", "Month corresponding to period"],
    ["hourly_earnings", "float", "U.S. Dollars per hour", "Continuous Numeric Variable", "Nonnegative dollar amount", "0 - missing means earnings were not reported", "BLS average hourly earnings, raw observed range was $14.15-$53.90"],
    ["month", "int", "Calender Month #", "Not categorical", "1-12", "0 - missing would mean period could not be converted", "Derived from period; 1=January through 12=December"],
    ["date", "date", "Calender Date", "Not Categorical", "January 2015-December 2025", "0 - missing would mean a valid date could not be created", "Created from year and month, first day represents the monthly timestamp"]
], columns=["Variable", "Type", "Units", "Factor levels", "Valid range", "Missingness", "Notes"])


data_dictionary

,Variable,Type,Units,Factor levels,Valid range,Missingness,Notes
0,series_id,string,Identifier,8 selected BLS CES series IDs,One of the 8 selected BLS CES series IDs,0 - missing means the BLS series is unidentified,BLS identifier for each industry series
1,industry,string/category,Industry Category/Categorical Variable,8 selected industries,One of the 8 selected industries,0 - missing means the industry is unidentified,Industry name is connected with series_id
2,year,integer,year,Calender Year,2015-2025 for this project,0 - missing means the observation year is unknown,Presented as text and converted to integer
3,period,string/category,Monthly Period Code,M01-M12,M01-M12,0 - missing means the monthly period is unknown,Original BLS month code - M01=January through ...
4,period_name,string/category,Calender Month,January-December,January-December,0 - missing means the month name is unknown,Month corresponding to period
5,hourly_earnings,float,U.S. Dollars per hour,Continuous Numeric Variable,Nonnegative dollar amount,0 - missing means earnings were not reported,"BLS average hourly earnings, raw observed rang..."
6,month,int,Calender Month #,Not categorical,1-12,0 - missing would mean period could not be con...,Derived from period; 1=January through 12=Dece...
7,date,date,Calender Date,Not Categorical,January 2015-December 2025,0 - missing would mean a valid date could not ...,"Created from year and month, first day represe..."


### Data Dictionary Notes

The audit found no observed values that contradict the expected structure of
the variables. All period values fall within M01 through M12, all observations
have one of the eight expected BLS series IDs and industry labels, and all
earnings values can be interpreted as numeric values.

The observed earnings range of $14.15 to $53.90 is reported as descriptive
information rather than as the variable's valid range based on findings. Values outside that observed range are not invalid however, because average hourly earnings could change over time.

The month and date variables are not present in the raw BLS data. They will be
created during cleaning while the original year and period variables are
retained for it to be traced back to the original data.

## 3. Cleaning Log

## 3. Cleaning Log

The audit showed that the BLS dataset is well structured and does not require
a large amount of cleaning. Therefore, cleaning is limited to correcting the data types of two variables and creating two variables needed for future analysis.

A separate cleaned copy is created below so that the raw data remain unchanged.

In [22]:
clean_bls = raw_bls.copy()
print("Raw shape:", raw_bls.shape)
print("Cleaned starting shape:", clean_bls.shape)

Raw shape: (1056, 6)
Cleaned starting shape: (1056, 6)


### Decision 1: Convert **year** to Integer

The BLS API returned year as text. Because year is a numeric calendar value
and will be used to create dates, I converted it to an integer. The audit
confirmed that all values could be converted successfully.

In [23]:
year_cells_affected = clean_bls["year"].notna().sum()
clean_bls["year"] = pd.to_numeric(clean_bls["year"],  errors="raise").astype(int)
print("Cells affected:", year_cells_affected)
print("New year dtype:", clean_bls["year"].dtype)

Cells affected: 1056
New year dtype: int64


### Decision 2: Convert **hourly_earnings** to Numeric

The BLS API returned **hourly_earnings** as text even though the variable
represents a dollar amount. Numeric data types are required for later
calculations such as wage growth and inflation adjustment. The audit confirmed
that all earnings values could be converted successfully.

In [24]:
earnings_cells_affected = (clean_bls["hourly_earnings"].notna().sum())
clean_bls["hourly_earnings"] = pd.to_numeric(clean_bls["hourly_earnings"], errors="raise")

print("Cells affected:", earnings_cells_affected)

print("New hourly_earnings dtype:", clean_bls["hourly_earnings"].dtype)

Cells affected: 1056
New hourly_earnings dtype: float64


### Decision 3: Create Numeric **month**

The original BLS **period** variable uses codes M01 through M12. I created a
numeric **month** variable by removing the "M" prefix and converting the remaining two digits to an integer. The original **period** variable is kept, so no original information is lost.

In [25]:
clean_bls["month"] = (clean_bls["period"].str.replace("M", "", regex=False).astype(int))
month_cells_created = clean_bls["month"].notna().sum()
print("Month values created:", month_cells_created)
print("Month range:", clean_bls["month"].min(), "to", clean_bls["month"].max())

Month values created: 1056
Month range: 1 to 12


### Decision 4: Create **date**

For later time-series analysis, I created a single chronological **date**
variable from **year** and **month**. Each observation is assigned the first day of its month because the BLS observations represent monthly rather than the daily earnings. The original **year, period, and period_name** variables are kept, so the created date can be traced back to the original values.

In [59]:
clean_bls["date"] = pd.to_datetime({"year": clean_bls["year"], "month": clean_bls["month"], "day": 1})
date_cells_created = clean_bls["date"].notna().sum()

print("Date Values Created:", date_cells_created)

print("Date Range:", clean_bls["date"].min(), "to", clean_bls["date"].max())

Date Values Created: 1056
Date Range: 2015-01-01 00:00:00 to 2025-12-01 00:00:00


### Cleaned Data (First 5 Rows)

After the four decisions:

In [27]:
display(clean_bls.head())
print("\nCleaned dataset shape:")
print(clean_bls.shape)
print("\nCleaned data types:")
print(clean_bls.dtypes)

,series_id,industry,year,period,period_name,hourly_earnings,month,date
0,CES2000000003,Construction,2024,M12,December,38.94,12,2024-12-01
1,CES2000000003,Construction,2024,M11,November,38.77,11,2024-11-01
2,CES2000000003,Construction,2024,M10,October,38.69,10,2024-10-01
3,CES2000000003,Construction,2024,M09,September,38.55,9,2024-09-01
4,CES2000000003,Construction,2024,M08,August,38.34,8,2024-08-01



Cleaned dataset shape:
(1056, 8)

Cleaned data types:
series_id                  object
industry                   object
year                        int64
period                     object
period_name                object
hourly_earnings           float64
month                       int64
date               datetime64[ns]
dtype: object


### Post-Cleaning Validation

I rechecked the cleaned dataset to verify that the transformations did not
introduce missing values, duplicate observation keys, invalid months, or
changes to the number of observations.

In [28]:
print("Rows in raw data:", len(raw_bls))
print("Rows in cleaned data:", len(clean_bls))
print("Missing values in cleaned data:")
display(clean_bls.isna().sum().to_frame(name="missing_count"))

print("Exact duplicate rows:", clean_bls.duplicated().sum())
print("Duplicate series-year-period keys:", clean_bls.duplicated(subset=["series_id", "year", "period"]).sum())

print("Invalid month values:", (~clean_bls["month"].between(1, 12)).sum())
print("Invalid dates:", clean_bls["date"].isna().sum())

Rows in raw data: 1056
Rows in cleaned data: 1056
Missing values in cleaned data:


,missing_count
series_id,0
industry,0
year,0
period,0
period_name,0
hourly_earnings,0
month,0
date,0


Exact duplicate rows: 0
Duplicate series-year-period keys: 0
Invalid month values: 0
Invalid dates: 0


In [29]:
print("Raw dataset still has:", raw_bls.shape[0], "rows and", raw_bls.shape[1], "columns")
print("Raw year dtype:", raw_bls["year"].dtype)
print("Raw hourly_earnings dtype:", raw_bls["hourly_earnings"].dtype)

Raw dataset still has: 1056 rows and 6 columns
Raw year dtype: object
Raw hourly_earnings dtype: object


### Cleaning Log

The table below shows each change made to the raw data. Because the audit
found no missing observations, duplicate records, invalid monthly periods,
sentinel earnings values, or incomplete industry-year combinations, no rows
were deleted and no values were imputed.

All original data is kept if the need for tracing back, or restarting occurs.

In [60]:
cleaning_log = pd.DataFrame([
    [1, "year", "Converted text to integer", year_cells_affected, "Year was text, but numeric year is needed for date creation and chronological analysis", "Nothing - values are unchanged and original text in raw_bls"],
    [2, "hourly_earnings", "Converted text to numeric", earnings_cells_affected, "Earnings were text, but numeric values are needed for wage-growth and inflation-adjustment calculations", "Nothing - values are unchanged and original text in raw_bls"],
    [3, "month", "Created numeric month from period", month_cells_created, "Numeric month is needed to for a date variable", "Nothing - period is kept and month is a new variable"],
    [4, "date", "Created date from year and month", date_cells_created, "A date variable is needed to merge and analyze monthly observations over time", "Nothing - year, month, and period are kept, day 1 is only a monthly timestamp"]
], columns=["#", "Column", "Change made", "Rows/cells affected", "Why", "What is lost"])

cleaning_log

,#,Column,Change made,Rows/cells affected,Why,What is lost
0,1,year,Converted text to integer,1056,"Year was text, but numeric year is needed for ...",Nothing - values are unchanged and original te...
1,2,hourly_earnings,Converted text to numeric,1056,"Earnings were text, but numeric values are nee...",Nothing - values are unchanged and original te...
2,3,month,Created numeric month from period,1056,Numeric month is needed to for a date variable,Nothing - period is kept and month is a new va...
3,4,date,Created date from year and month,1056,A date variable is needed to merge and analyze...,"Nothing - year, month, and period are kept, da..."


## 4. Row and Column Accounting

No rows or original columns were removed during cleaning. Two new columns, month and date, were added for chronological analysis.

In [63]:
raw_rows = len(raw_bls)
clean_rows = len(clean_bls)
exact_duplicates_removed = 0
near_duplicates_removed = 0
other_rows_removed = 0
raw_columns = raw_bls.shape[1]
clean_columns = clean_bls.shape[1]
columns_dropped = len(set(raw_bls.columns) - set(clean_bls.columns))
columns_added = len(set(clean_bls.columns) - set(raw_bls.columns))

print(f"Rows in raw file - {raw_rows}")
print(f"Rows removed as exact duplicates - {exact_duplicates_removed}")
print(f"Rows removed as near-duplicates - {near_duplicates_removed}")
print(f"Rows removed for other reasons - {other_rows_removed}")
print(f"Rows in cleaned file - {clean_rows}")
print(f"Columns in raw file - {raw_columns}")
print(f"Columns dropped - {columns_dropped}")
print(f"Columns added - {columns_added}")
print(f"Columns in cleaned file - {clean_columns}")

Rows in raw file - 1056
Rows removed as exact duplicates - 0
Rows removed as near-duplicates - 0
Rows removed for other reasons - 0
Rows in cleaned file - 1056
Columns in raw file - 6
Columns dropped - 0
Columns added - 2
Columns in cleaned file - 8


### Accounting Summary

All 1,056 raw observations were kept because the audit found no exact
duplicates, near-duplicates, missing records, or other invalid observations. No original columns were dropped.

Two new columns were added: month, which converts the BLS M01-M12 period
codes to numeric months, and date, which provides a standard monthly
timestamp for chronological analysis. Therefore, the dataset increased from
6 to 8 columns while still containing 1,056 rows.



## 5. Provenance Brief


The wage data used in this project came from the United States Bureau of Labor Statistics, also known as BLS, Current Employment Statistics program. BLS produces the data to measure factors like employment, hours, and earnings across many industries in the United States. The data was retrieved through the BLS public API for eight selected industries and cover from January 2025 to December 2025.

For this project, the dataset contains monthly average hourly earnings for employees in Construction, Manufacturing, Trade/Transportation/Utilities, Information, Financial Activites, Professional/Business Services, Private Education/Health Services, and Leisure/Hospitality. Each record identifies an industry, year, month, and average hourly earnings value.

This data can be used to compare how average hourly earnings have changed across the selected industries over time. However, the earning values may not account for inflation, and do not measure changes in the sector. Inflation data will be incorporated later in the project for further analysis on this subject. The data also represents industry averages, so they are not necessarily the earnings of every single worker within an industry.


## Self-Graded Rubric

| **Criterion** | **Wt** | **The short version** | **Self-score** |
|---|---:|---|---:|
| Diagnosis & data dictionary | ×2 | Systematic audit, quantified problems, and what was checked for and not found | 5/5 |
| Cleaning execution | ×2 | Correct, proportionate, alternatives considered | 5/5 |
| **Cleaning log** | **×3** | Counts, reasons, what was lost, reversible | **5/5** |
| Provenance Brief | ×1 | A non-specialist understands it | 5/5 |
| Tidy structure & reproducibility | ×2 | Tidy, runs without errors, README present | 5/5 |

**Weighted Self-Score: 50/50**

In [65]:
raw_bls.to_csv("bls_hourly_earnings_raw.csv", index=False)